# Lecture: Comparing Distributions Across Groups

Compare the distribution of trip duration across categorical groups using bike-share records. We will investigate station-return patterns and membership categories, then examine how unequal group sizes change what different plots communicate.


## Learning goals

- Compare center, spread, shape, overlap, and unusual values.
- Use count histograms, within-group percentage histograms, box plots, and violin plots.
- Explain why a standard box plot does not reveal the number of observations.
- Distinguish a display zoom from removing data.
- Apply the six-step workflow and support conclusions with numerical evidence.


## Data and context

Use the September 2025 Jersey City trip-history file from [Citi Bike System Data](https://citibikenyc.com/system-data). The fixed local file retains seven source columns and all 116,071 records from that monthly file. Each row is a trip, not a unique rider. The JC file covers the Jersey City/Hoboken service area; it is not the full New York City system.

Source archive: [September 2025 JC trips](https://s3.amazonaws.com/tripdata/JC-202509-citibike-tripdata.csv.zip). See the local data README for preparation and the [data-use policy](https://citibikenyc.com/data-sharing-policy). This is an independent teaching analysis.

We analyze **elapsed trip duration**, not time of day. Duration has a meaningful zero and differences measured in minutes; unlike a clock reading, it does not wrap around at midnight. A trip lasting 20 minutes is twice as long as one lasting 10 minutes. Recorded duration can include stops or time before a bicycle is returned, not just active cycling.


### Data dictionary

| Feature | Meaning |
| --- | --- |
| `ride_id` | Trip identifier |
| `started_at`, `ended_at` | Recorded start and end timestamps |
| `start_station_id`, `end_station_id` | Station identifiers; not quantitative measurements |
| `member_casual` | Recorded membership category: member or casual |
| `rideable_type` | Recorded bicycle type |

We derive `duration_min` and, for records with both station IDs, `return_group`. A same-station trip does not establish a recreational purpose or a particular route.


## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display?
4. **Check:** Does the result make sense in context?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?


## Prepare the notebook

Seaborn works with Pandas DataFrames to make statistical graphics. Matplotlib adds titles, labels, and display limits.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Research question 1: Returning to the starting station

## Step 1: Question

**How does trip duration differ between trips ending at their starting station and trips ending at a different station?**


**Predict which group will have the longer typical duration and whether one group will have many more trips. Explain your reasoning.**


Your prediction:


## Step 2: Data

Load the fixed local file. Parse timestamps as dates and keep station IDs as strings.


In [ ]:
trips_df = pd.read_csv(
    "data/citibike_jc_202509.csv",
    parse_dates=["started_at", "ended_at"],
    dtype={"start_station_id": "string", "end_station_id": "string"}
)


In [ ]:
trips_df.head()


In [ ]:
trips_df.tail()


In [ ]:
trips_df.shape


In [ ]:
trips_df.info()


In [ ]:
trips_df.isna().sum()


Subtract timestamps to obtain elapsed time. `.dt.total_seconds()` retains complete elapsed days as well as seconds; divide by 60 to express the result in minutes.


In [ ]:
trips_df["duration_min"] = (trips_df["ended_at"] - trips_df["started_at"]).dt.total_seconds() / 60
trips_df[["started_at", "ended_at", "duration_min"]].head()


In [ ]:
trips_df["duration_min"].describe()


Keep positive, known durations. This source month has no missing or nonpositive durations, but the rule states the intended scope. Very long trips remain in the analysis; a long duration alone does not prove an error.


In [ ]:
duration_df = trips_df.loc[trips_df["duration_min"].notna() & (trips_df["duration_min"] > 0)].copy()
duration_df.shape


For this question, exclude records without either station ID: an unknown destination is not evidence of a different destination. Membership analysis will reuse all valid-duration trips.


In [ ]:
station_df = duration_df.dropna(subset=["start_station_id", "end_station_id"]).copy()
station_df.shape


In [ ]:
station_df["return_group"] = "Different stations"
station_df.loc[station_df["start_station_id"] == station_df["end_station_id"], "return_group"] = "Same station"
station_df["return_group"].value_counts()


## Step 3: Evidence


In [ ]:
station_df.groupby("return_group")["duration_min"].describe().round(2)


### Inspect the full range before zooming

Use a full-range histogram to see the long right tail. Then compare groups in a 0–60 minute view. A display limit is not a filter: all summary statistics and histogram normalization still include longer trips.


In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=station_df, x="duration_min", bins=80, color="darkseagreen")
plt.title("Station-comparison trips: full duration range")
plt.xlabel("Elapsed trip duration (minutes)")
plt.ylabel("Number of trips")
plt.tight_layout()
plt.show()


In [ ]:
duration_bins = list(range(0, int(duration_df["duration_min"].max()) + 3, 2))
print("Histogram coverage in minutes:", duration_bins[0], "to", duration_bins[-1])


In [ ]:
station_df.assign(over_60_min=station_df["duration_min"] > 60).groupby("return_group")["over_60_min"].agg(["sum", "mean"])


In the table above, `sum` counts trips beyond the display window and `mean` gives their fraction within each group.


In [ ]:
station_return_order = ['Different stations', 'Same station']
station_return_colors = dict(zip(station_return_order, ["darkseagreen", "mediumpurple"]))
station_return_colors


### Count histogram: how many trips?

Use the same two-minute bins for both groups. Count histograms retain differences in sample size, but peak height also depends on how concentrated each distribution is. Read the counts alongside the group-size table.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=station_df, x="duration_min", hue="return_group", hue_order=station_return_order,
             palette=station_return_colors, bins=duration_bins, stat="count", element="step", fill=False)
plt.xlim(0, 60)
plt.title("Trip duration by station return: counts (0–60 minute view)")
plt.xlabel("Elapsed trip duration (minutes)")
plt.ylabel("Number of trips per two-minute bin")
plt.tight_layout()
plt.show()


### Percentage histogram: what fraction of each group?

`stat="percent"` and `common_norm=False` normalize each group separately. The complete histogram for each group totals 100%; the displayed 0–60 minute portion can total less because longer trips remain in the denominator. This comparison removes the effect of unequal group sizes while retaining distribution shape.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=station_df, x="duration_min", hue="return_group", hue_order=station_return_order,
             palette=station_return_colors, bins=duration_bins, stat="percent", common_norm=False,
             element="step", fill=False)
plt.xlim(0, 60)
plt.title("Trip duration by station return: within-group percentages")
plt.xlabel("Elapsed trip duration (minutes; 0–60 minute view)")
plt.ylabel("Percent of group per two-minute bin")
plt.tight_layout()
plt.show()


### Box plot: center and spread

The box spans the middle 50% (Q1 to Q3); its line is the median. Default whiskers extend to observations within 1.5 interquartile ranges of the box, with more distant values plotted individually. Those points are not automatically errors. Equal box widths do not encode group size. The statistics use every included trip, even where the display is zoomed.


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=station_df, x="duration_min", y="return_group", order=station_return_order,
            hue="return_group", hue_order=station_return_order, palette=station_return_colors, legend=False, fliersize=2)
plt.xlim(0, 60)
plt.title("Trip duration by station return: equal-width box plots")
plt.xlabel("Elapsed trip duration (minutes; 0–60 minute view)")
plt.ylabel("")
plt.tight_layout()
plt.show()


### Violin plot: smoothed shape

A violin adds an estimated density to an interior box summary. `cut=0` stops the curve at observed extremes; `density_norm="area"` gives equal areas to the complete violins, so width is not a count. Smoothing and long tails influence the shape. Like the box plot, this display does not communicate group sizes directly.


In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(data=station_df, x="duration_min", y="return_group", order=station_return_order,
               hue="return_group", hue_order=station_return_order, palette=station_return_colors,
               legend=False, inner="box", cut=0, gridsize=1024, density_norm="area", common_norm=False)
plt.xlim(0, 60)
plt.title("Trip duration by station return: violin plots")
plt.xlabel("Elapsed trip duration (minutes; 0–60 minute view)")
plt.ylabel("")
plt.tight_layout()
plt.show()


**Which plot best reveals the imbalance? Which makes the duration difference clearest? Compare the median, middle 50%, right tail, and overlap. Does a taller count-histogram peak imply a longer typical trip?**


## Step 4: Check

Do the calculated durations agree with the timestamps? Do the station labels agree with station-ID equality? Read the unusually long records before deciding what they mean.


In [ ]:
station_df.nlargest(5, "duration_min")[["started_at", "ended_at", "duration_min", "return_group"]]


In [ ]:
pd.crosstab(station_df["start_station_id"] == station_df["end_station_id"], station_df["return_group"])


**Would a 25-hour recorded duration necessarily mean 25 hours of continuous cycling? What would make you suspicious of the calculation or category assignment? Why must a count histogram and a percentage histogram have different y-axis labels?**


## Step 5: Conclusion

**Describe the difference in typical duration and spread using numerical evidence. How does it compare with your prediction?**


## Step 6: Limitation

**Why can’t we conclude that returning to the same station causes a longer trip, or that these are all recreational rides? How might the missing station IDs affect the comparison?**


# Research question 2: Members and casual riders

## Step 1: Question

**How does trip duration differ between member trips and casual-rider trips?**


**Predict which group will have a longer median and which group will contain more trips. Explain your reasoning.**


Your prediction:


## Step 2: Data

Reuse `duration_df`, including trips with missing station IDs: this question needs only duration and `member_casual`. Membership is a category recorded for the trip, not a unique-person identifier.


In [ ]:
duration_df["member_casual"].value_counts(dropna=False)


## Step 3: Evidence

Repeat the same distribution comparisons, with identical bins and a 0–60 minute view. Keep all valid durations in the summaries and normalization.


In [ ]:
duration_df.groupby("member_casual")["duration_min"].describe().round(2)


In [ ]:
duration_df.assign(over_60_min=duration_df["duration_min"] > 60).groupby("member_casual")["over_60_min"].agg(["sum", "mean"])


In [ ]:
membership_order = ['member', 'casual']
membership_colors = dict(zip(membership_order, ["darkseagreen", "mediumpurple"]))
membership_colors


### Count histogram: how many trips?

Use the same two-minute bins for both groups. Count histograms retain differences in sample size, but peak height also depends on how concentrated each distribution is. Read the counts alongside the group-size table.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=duration_df, x="duration_min", hue="member_casual", hue_order=membership_order,
             palette=membership_colors, bins=duration_bins, stat="count", element="step", fill=False)
plt.xlim(0, 60)
plt.title("Trip duration by membership: counts (0–60 minute view)")
plt.xlabel("Elapsed trip duration (minutes)")
plt.ylabel("Number of trips per two-minute bin")
plt.tight_layout()
plt.show()


### Percentage histogram: what fraction of each group?

`stat="percent"` and `common_norm=False` normalize each group separately. The complete histogram for each group totals 100%; the displayed 0–60 minute portion can total less because longer trips remain in the denominator. This comparison removes the effect of unequal group sizes while retaining distribution shape.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=duration_df, x="duration_min", hue="member_casual", hue_order=membership_order,
             palette=membership_colors, bins=duration_bins, stat="percent", common_norm=False,
             element="step", fill=False)
plt.xlim(0, 60)
plt.title("Trip duration by membership: within-group percentages")
plt.xlabel("Elapsed trip duration (minutes; 0–60 minute view)")
plt.ylabel("Percent of group per two-minute bin")
plt.tight_layout()
plt.show()


### Box plot: center and spread

The box spans the middle 50% (Q1 to Q3); its line is the median. Default whiskers extend to observations within 1.5 interquartile ranges of the box, with more distant values plotted individually. Those points are not automatically errors. Equal box widths do not encode group size. The statistics use every included trip, even where the display is zoomed.


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=duration_df, x="duration_min", y="member_casual", order=membership_order,
            hue="member_casual", hue_order=membership_order, palette=membership_colors, legend=False, fliersize=2)
plt.xlim(0, 60)
plt.title("Trip duration by membership: equal-width box plots")
plt.xlabel("Elapsed trip duration (minutes; 0–60 minute view)")
plt.ylabel("")
plt.tight_layout()
plt.show()


### Violin plot: smoothed shape

A violin adds an estimated density to an interior box summary. `cut=0` stops the curve at observed extremes; `density_norm="area"` gives equal areas to the complete violins, so width is not a count. Smoothing and long tails influence the shape. Like the box plot, this display does not communicate group sizes directly.


In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(data=duration_df, x="duration_min", y="member_casual", order=membership_order,
               hue="member_casual", hue_order=membership_order, palette=membership_colors,
               legend=False, inner="box", cut=0, gridsize=1024, density_norm="area", common_norm=False)
plt.xlim(0, 60)
plt.title("Trip duration by membership: violin plots")
plt.xlabel("Elapsed trip duration (minutes; 0–60 minute view)")
plt.ylabel("")
plt.tight_layout()
plt.show()


**Which differences remain after normalizing within each group? Is the separation as strong as in the station comparison? What information would you lose if shown only the box plots?**


## Step 4: Check

**Do the histogram peaks fall in plausible locations relative to the medians and quartiles? Does normalization change the duration values, or only the vertical scale? Why are some long trips absent from the displayed window but still represented in the summaries?**


## Step 5: Conclusion

**Answer the question with medians and a statement about overlap. Compare your conclusion with your prediction.**


## Step 6: Limitation

**Why does this not establish that buying a membership makes a person take shorter trips? What other information might explain the pattern?**


## Choosing a plot

| Plot | Especially useful for | What it does not show well |
| --- | --- | --- |
| Count histogram with common bins | Numbers of observations and detailed shape | Unequal group sizes can obscure a smaller group; peak height also depends on concentration |
| Within-group percentage histogram | Comparing shapes despite unequal sample sizes | Original group sizes unless separately labeled |
| Equal-width box plot | Median, middle 50%, and unusual values | Sample size, modes, and detailed shape |
| Equal-area violin plot | Smoothed shape with an interior summary | Sample size; shape depends on density estimation |

Always state the observation unit, group sizes, units, denominator, and any display limits. A difference in group centers can coexist with considerable overlap.
